In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [4]:
dataset = pd.read_csv('Churn_Modelling.csv', header=0)
print(dataset.shape)
dataset.head()

(10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
#preprocessing the data
dataset.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)
dataset.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
#only geography and gender are categorical features
print("geography: ",dataset['Geography'].unique())
print("gender: ",dataset['Gender'].unique())

#preprocessing the categorical features
gender_encoder = LabelEncoder()
dataset['Gender'] = gender_encoder.fit_transform(dataset['Gender'])

from sklearn.preprocessing import OneHotEncoder
geography_encoder = OneHotEncoder(sparse_output=False)
geography_new = pd.DataFrame(geography_encoder.fit_transform(dataset[['Geography']]),columns=geography_encoder.get_feature_names_out(['Geography']))
print(geography_new.head())

geography:  <ArrowStringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str
gender:  <ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0
1               0.0                0.0              1.0
2               1.0                0.0              0.0
3               1.0                0.0              0.0
4               0.0                0.0              1.0


In [7]:
#concat all data
dataset = pd.concat([dataset.drop('Geography', axis=1),geography_new],axis=1)
dataset.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [8]:
# saving the encoders
with open('gender_encoder.pkl', 'wb') as file:
    pickle.dump(gender_encoder, file)

with open('geography_encoder.pkl', 'wb') as file:
    pickle.dump(geography_encoder, file)


In [9]:
# X and Y data
X = dataset.drop('Exited', axis=1)
Y = dataset['Exited']

# splitting the data into training and testing data
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

#feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [11]:
print("X_train shape: ",X_train.shape)
print("X_test shape: ",X_test.shape)
print("Y_train shape: ",Y_train.shape)
print("Y_test shape: ",Y_test.shape)

X_train shape:  (8000, 12)
X_test shape:  (2000, 12)
Y_train shape:  (8000,)
Y_test shape:  (2000,)


# ANN MODEL

In [12]:
import tensorflow as tf

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [14]:
# Creating the model
model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)), #HL1
    Dense(8, activation='relu'), #HL2
    Dense(1, activation='sigmoid') #output layer
])

model.summary()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 353 (1.38 KB)

 Trainable params: 353 (1.38 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
#compiling the model

from tensorflow.keras.optimizers import Adam
adam_opt = Adam(learning_rate=0.01)

model.compile(optimizer=adam_opt, loss='binary_crossentropy', metrics=['accuracy'])

In [16]:
# creating log directory and creating callbacks for early stopping and tensorboard

import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [17]:
# training the model
history = model.fit(X_train, Y_train, epochs=100, validation_data=(X_test, Y_test), callbacks=[early_stopping, tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 738us/step - accuracy: 0.8311 - loss: 0.4055 - val_accuracy: 0.8540 - val_loss: 0.3578
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 472us/step - accuracy: 0.8543 - loss: 0.3576 - val_accuracy: 0.8560 - val_loss: 0.3463
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - accuracy: 0.8546 - loss: 0.3492 - val_accuracy: 0.8520 - val_loss: 0.3499
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 467us/step - accuracy: 0.8554 - loss: 0.3490 - val_accuracy: 0.8610 - val_loss: 0.3409
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 476us/step - accuracy: 0.8593 - loss: 0.3454 - val_accuracy: 0.8665 - val_loss: 0.3366
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - accuracy: 0.8580 - loss: 0.3431 - val_accuracy: 0.8580 - val_loss: 0.3388
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 468us/step - accuracy: 0.8602 - loss: 0.3409 - val_accuracy: 0.8630 - val_loss: 0.3360
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 468us/step - accuracy: 0.8600 - loss: 0

In [18]:
# saving the model
model.save('churn_model.h5')

In [19]:
## Load Tensorboard Extension
%load_ext tensorboard

In [20]:
%tensorboard --logdir logs